# M4 선형 회귀 — 실습 (W6)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 세트피스(점 3개)의 **후보 대결·경사하강·바닥의 기울기 0**을 코드로 재현하고 sklearn 검산으로 일치를 확인한다 ⭐
2. 당뇨 데이터로 단순/다변수 회귀를 적합하고 지표 4종(MAE/MSE/RMSE/R²)을 해석한다
3. **학습률 4종 대결**(0.1/0.5/1.0/5.0)로 느림·수렴·발산을 눈으로 확인한다 ⭐

**7단계 멘탈모델 초점:** 손실 + 최적화 — ML의 심장

## Part A. 세트피스 — 세상에서 가장 작은 회귀 ⭐
점 (1,2), (2,2), (3,5), 모델 `ŷ = w·x`(원점 통과). 먼저 종이에서 w=1, 2, 1.5의 MSE를 계산해 보고, 경사하강 두 걸음을 밟아 본 뒤 코드로 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산

x = np.array([1.0, 2, 3])                              # 점 3개의 x
t = np.array([2.0, 2, 5])                              # 정답 y
for w in (1.0, 2.0, 1.5):                              # 1라운드 — 후보 대결
    mse = ((w * x - t) ** 2).___()                     # ✍️ 빈칸: MSE = 오차 제곱의 무엇?
    print(f'w={w}: MSE = {round(mse, 4)}')             # 1.6667 / 1.6667(동점!) / 0.5

In [ ]:
w = 1.0                                                # 2라운드 — 경사하강, 나쁜 시작
for step in range(1, 4):                               # 세 걸음
    err = w * x - t                                    # 오차(예측 − 정답)
    grad = (2 / 3) * (err * ___).sum()                 # ✍️ 빈칸: 기울기 = (2/n)Σ(오차·무엇?)
    w = w - 0.1 * ___                                  # ✍️ 빈칸: 기울기 반대 방향으로 한 걸음(학습률 0.1)
    print(f'걸음{step}: 기울기 {round(grad, 4)} → w = {round(w, 4)}')  # 1.4667 → 1.4978 → 1.4999

err15 = 1.5 * x - t                                    # 바닥(w=1.5)에서의 오차
print('w=1.5의 기울기:', round((2 / 3) * (err15 * x).sum(), 6))  # 정확히 0 — 바닥의 증거!

from sklearn.linear_model import LinearRegression      # sklearn 검산
lin0 = LinearRegression().fit(x.reshape(-1, 1), t)     # 절편 자유로 적합해도
print('sklearn w, b:', round(lin0.coef_[0], 4), ',', round(lin0.intercept_, 4))  # 1.5, 0.0 — 일치

> **검산 포인트:** 후보 대결 — w=1과 w=2가 정확히 **동점(5/3)**, w=1.5가 **0.5로 최소**. 경사하강 — 1 → **1.467** → **1.498** → 1.4999(두세 걸음에 도착). **바닥(w=1.5)의 기울기 = 정확히 0** — 더 내려갈 곳이 없다는 증거. sklearn도 w=1.5(이 데이터는 b도 0). **학습의 뼈대 = 좋음을 숫자로(손실) + 그 숫자를 줄이는 방향으로 조금씩(경사하강).**

## Part B. 단순 선형회귀 — 당뇨 데이터, BMI 하나
실전 데이터에서 직선 하나로 요약해 봅니다. w의 부호가 곧 해석입니다.

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import load_diabetes             # 당뇨 진행도(회귀 데이터)
from sklearn.model_selection import train_test_split   # 분할(M2a)

dia = load_diabetes()                                  # 442명, 특징 10개
X, y = dia.data, dia.target
X_bmi = X[:, [___]]                                    # ✍️ 빈칸: BMI 특징의 index(0부터 셋째)
Xtr, Xte, ytr, yte = train_test_split(X_bmi, y, test_size=0.3, random_state=42)

lin = LinearRegression().fit(Xtr, ytr)                 # 최적 w, b 계산
print('기울기 w:', round(lin.coef_[0], 1), '| 절편 b:', round(lin.intercept_, 1))  # 988.4 / 151.0

plt.scatter(Xtr, ytr, alpha=0.5, label='data')         # 데이터 산점도
xs = np.linspace(Xtr.min(), Xtr.max(), 100).reshape(-1, 1)
plt.plot(xs, lin.predict(xs), 'r-', linewidth=2, label='fit line')  # 요약된 직선
plt.xlabel('BMI (normalized)'); plt.ylabel('disease progression')   # 축(영어)
plt.legend(); plt.title('Simple linear regression (BMI)')
plt.show()

> **관찰:** w = **988.4**(양수) — "BMI↑ → 진행도 예측↑"가 w의 부호에서 바로 읽힘. 수백 개의 점이 **숫자 두 개(w, b)로 요약**됐습니다 — KNN이라면 442명을 전부 들고 다녀야 했던 일.

## Part C. 다변수 회귀 + 지표 4종 (M2b 실전 투입)
특징 10개를 모두 쓰고, M2b에서 배운 회귀지표로 채점합니다.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 회귀지표(M2b)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)  # 전체 특징으로
model = LinearRegression().fit(Xtr, ytr)               # 학습
y_pred = model.predict(___)                            # ✍️ 빈칸: 시험 특징으로 예측

mae = mean_absolute_error(yte, y_pred)                 # 평균 절대 오차
mse = mean_squared_error(yte, y_pred)                  # 평균 제곱 오차
rmse = np.sqrt(___)                                    # ✍️ 빈칸: RMSE = √(무엇?)
r2 = r2_score(yte, y_pred)                             # 결정계수
print('MAE :', round(mae, 1))                          # 41.9
print('RMSE:', round(rmse, 1))                         # 53.1 — 원래 단위의 오차
print('R^2 :', round(r2, 3))                           # 0.477 — 변동의 약 48% 설명

plt.scatter(yte, y_pred, alpha=0.5)                    # 예측 vs 실제
lims = [yte.min(), yte.max()]
plt.plot(lims, lims, 'r--', label='perfect')           # 완벽 예측 대각선
plt.xlabel('actual'); plt.ylabel('predicted')          # 축(영어)
plt.legend(); plt.title('Predicted vs Actual')
plt.show()

> **관찰:** MAE 41.9 / RMSE 53.1 / R² 0.477 — 평균 ±42~53쯤 빗나가고 변동의 약 48%를 설명(의료 데이터로는 의미 있는 수준). 예측-실제 그림에서 추세는 따라가되 흩어짐이 큼 — **숫자와 그림을 함께** 보는 습관(M2b).

## Part D. 경사하강 관찰 + 학습률 4종 대결 ⭐
BMI 한 특징으로 w, b를 0에서 시작해 손실이 줄어드는 과정을 보고, 학습률을 0.1 / 0.5 / 1.0 / 5.0으로 바꿔 **세 가지 운명**(느림/수렴/발산)을 확인합니다.

In [ ]:
xg = X[:, 2]                                           # BMI 특징(1차원)
tg = y                                                 # 정답
n = len(xg)                                            # 샘플 수

def run_gd(lr, steps=50):                              # 경사하강 실행기
    w, b, losses = 0.0, 0.0, []                        # 초기값
    for _ in range(steps):                             # steps번 갱신
        err = w * xg + b - tg                          # 오차
        w -= lr * (2 / n) * np.sum(err * xg)           # w를 내리막으로
        b -= lr * (2 / n) * np.sum(err)                # b를 내리막으로
        losses.append(np.mean(err ** 2))               # 손실 기록
    return losses

base = run_gd(0.5)                                     # 기준: lr=0.5
print('lr=0.5: 손실', round(base[0]), '→', round(base[-1], 1))   # 29074 → 5523.9

plt.figure(figsize=(8, 4.5))
for lr_try in (0.1, 0.5, 1.0, ___):                    # ✍️ 빈칸: 폭발을 볼 큰 학습률(본문의 그 수)
    plt.semilogy(run_gd(lr_try), label=f'lr = {lr_try}')  # 세로축 로그(폭발을 한 화면에)
plt.xlabel('iteration'); plt.ylabel('loss (MSE, log scale)')  # 축(영어)
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Learning rate: slow / converge / oscillate / explode')
plt.show()
for lr_try in (0.1, 0.5, 1.0, 5.0):                    # 최종 손실 표
    print(f'lr={lr_try}: 50걸음 후 손실 {run_gd(lr_try)[-1]:.4g}')

> **관찰:** 0.1 = 5841(느리지만 안정) · **0.5 = 5524(수렴)** · 1.0 = 28343(**진동** — 손실 곡선은 평평해 보이지만 파라미터가 골짜기 양쪽을 왕복하며 제자리 — 0.5 대비 크게 악화) · **5.0 = 7.6×10⁹⁷(폭발!)**. 보폭이 임계를 넘으면 골짜기를 건너뛰어 더 높은 곳에 떨어지기를 반복. **작으면 느림·적당하면 수렴·크면 발산** — 2학기(2학기 D1)에서 "학습률의 세 운명"으로 재회할 표입니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "점 (1,1),(2,3),(3,4)로 w=1과 1.5의 MSE를 내가 계산할 테니 채점해 줘."
- "경사하강 한 걸음을 내 숫자로 밟아 볼게 — 검산해 줘."
- "바닥에서 기울기가 0인 이유를 그릇 비유로 설명해 볼게 — 허점을 찔러 줘."
- "lr=1.0이 '진동'하는 이유를 보폭과 골짜기 폭으로 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 세트피스(점 3개)로 **후보 대결(1.67/1.67/0.5) → 경사하강(1→1.467→1.498) → 바닥의 기울기 0**을 완주하고 sklearn(w=1.5) 검산했다
2. 당뇨 데이터로 단순(w=988.4 해석)·다변수(MAE 41.9/RMSE 53.1/R² 0.477) 회귀를 적합했다
3. **학습률 4종 대결**로 느림/수렴/진동/폭발(10⁹⁷)을 눈으로 확인했다 — 2학기 "세 운명"의 복선

**스스로 점검**
- [ ] 후보 셋의 MSE를 종이에 재현할 수 있다
- [ ] 경사하강 한 걸음(w ← w − lr×기울기)을 숫자로 밟을 수 있다
- [ ] "바닥에서 기울기 0"의 뜻을 안다
- [ ] R² 0.477을 한 문장으로 해석할 수 있다
- [ ] 학습률의 세 거동을 실측 수치로 인용할 수 있다

**🔹심화 (선택)**
- 세트피스를 `ŷ = wx + b`로 확장해 보세요 — 기울기가 2개(∂w, ∂b)가 됩니다(2학기 D1이 이 확장을 그대로 밟습니다).
- `Ridge(alpha=...)`로 규제 세기를 바꿔 계수들이 어떻게 줄어드는지 관찰해 보세요.
- `PolynomialFeatures(degree=...)`로 다항 회귀를 만들어 차수↑에 따른 과적합(M2a)을 재현해 보세요.

**다음 시간(M5):** 가중합에 S자(시그모이드)를 씌우면 **분류**가 된다 — 손실은 왜 MSE에서 교체되는가?